# Handwritten Character Recognition — Exploration Notebook

This notebook walks through the project interactively:
1. Load and visualize the dataset (MNIST or EMNIST)
2. Inspect the CNN architecture
3. Train for a few epochs inline
4. Evaluate and visualize predictions
5. Try inference on your own drawn/uploaded image

Run `pip install -r requirements.txt` first if you haven't already.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.'))

import torch
import matplotlib.pyplot as plt
import numpy as np

import config
from src.dataset import get_dataloaders
from src.model import CharCNN

print('Device:', config.DEVICE)

## 1. Choose a dataset

Options: `mnist` (10 digit classes), `emnist_letters` (26 letter classes), `emnist_balanced` (47 classes: digits + upper + lowercase subset).

In [ ]:
DATASET_KEY = 'mnist'  # change to 'emnist_balanced' or 'emnist_letters' as desired

train_loader, val_loader, test_loader, num_classes, class_names = get_dataloaders(
    dataset_key=DATASET_KEY, batch_size=64, augment=True
)

print(f'Dataset: {DATASET_KEY}')
print(f'Classes ({num_classes}): {class_names}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Val samples:   {len(val_loader.dataset)}')
print(f'Test samples:  {len(test_loader.dataset)}')

## 2. Visualize sample images

In [ ]:
x, y = next(iter(train_loader))

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
mean, std = config.NORMALIZE_MEAN, config.NORMALIZE_STD

for i, ax in enumerate(axes.flatten()):
    img = x[i, 0].numpy() * std + mean
    ax.imshow(img, cmap='gray')
    ax.set_title(class_names[y[i].item()])
    ax.axis('off')

plt.suptitle(f'Sample images from {DATASET_KEY}')
plt.tight_layout()
plt.show()

## 3. Inspect the CNN architecture

In [ ]:
model = CharCNN(num_classes=num_classes, in_channels=config.IN_CHANNELS).to(config.DEVICE)
print(model)
print(f'\nTotal trainable parameters: {model.count_parameters():,}')

## 4. Quick training run (a few epochs, inline)

For full training with checkpointing, early stopping, and logging, prefer the command-line script:
```bash
python src/train.py --dataset mnist --epochs 15
```
This cell is for quick interactive experimentation only.

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 3  # bump this up for better accuracy

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x_batch, y_batch in tqdm(train_loader, desc=f'Epoch {epoch}'):
        x_batch, y_batch = x_batch.to(config.DEVICE), y_batch.to(config.DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += x_batch.size(0)

    print(f'Epoch {epoch}: loss={running_loss/total:.4f} acc={correct/total:.4f}')

## 5. Evaluate on the test set

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch, y_batch = x_batch.to(config.DEVICE), y_batch.to(config.DEVICE)
        preds = model(x_batch).argmax(1)
        correct += (preds == y_batch).sum().item()
        total += x_batch.size(0)

print(f'Test accuracy: {correct/total:.4f}')

## 6. Visualize predictions on test samples

In [ ]:
x, y = next(iter(test_loader))
with torch.no_grad():
    logits = model(x.to(config.DEVICE))
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1).cpu()
    confs = probs.max(dim=1)[0].cpu()

fig, axes = plt.subplots(3, 6, figsize=(14, 8))
for i, ax in enumerate(axes.flatten()):
    img = x[i, 0].numpy() * std + mean
    true_label = class_names[y[i].item()]
    pred_label = class_names[preds[i].item()]
    correct = y[i].item() == preds[i].item()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'T:{true_label} P:{pred_label} ({confs[i]:.0%})',
                 color='green' if correct else 'red', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Try your own handwritten image

Save a PNG/JPG of a single handwritten character to disk and point `IMAGE_PATH` to it.
Works best with dark strokes on a light background (or vice versa) — the preprocessing
pipeline auto-detects and inverts/centers/crops as needed.

In [ ]:
from src.predict import preprocess_char_image, array_to_tensor
from PIL import Image

IMAGE_PATH = '../outputs/sample_predictions_mnist.png'  # <-- change to your own image path

if os.path.exists(IMAGE_PATH):
    pil_img = Image.open(IMAGE_PATH).convert('L')
    raw = np.array(pil_img)
    processed = preprocess_char_image(raw)

    tensor = array_to_tensor(processed).to(config.DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    top5 = probs.argsort()[::-1][:5]

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(raw, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(processed, cmap='gray'); axes[1].set_title('Preprocessed (28x28)'); axes[1].axis('off')
    plt.show()

    print('Top-5 predictions:')
    for idx in top5:
        print(f'  {class_names[idx]}: {probs[idx]:.2%}')
else:
    print(f'No image found at {IMAGE_PATH} — update the path to try this section.')

## 8. Next steps: full word/sentence recognition with CRNN

See `src/crnn_model.py` and `src/crnn_train.py` for the sequence-modeling extension
that reads an entire word in one pass using a CNN + BiLSTM + CTC pipeline, instead of
segmenting into individual characters first. Try:
```bash
python src/crnn_train.py --epochs 10
```